# Human Labels Overview

Aggregates all human-labeled validation CSVs across benchmarks into a
single descriptive DataFrame.

In [1]:
from pathlib import Path
import pandas as pd

EVALS_ROOT = Path("../evals")

# Benchmark validation directories and the filename prefix to strip.
# Matches the BENCHMARKS config in all_evals_overview.ipynb.
VALIDATION_SOURCES = {
    "core_bench": {
        "validation_dir": EVALS_ROOT / "core_bench" / "validation",
        "prefix": "core_bench_",
        # Skip the old multi-label files
        "exclude_patterns": ["_multi", "_cheating"],
    },
    "swe_bench": {
        "validation_dir": EVALS_ROOT / "swe_bench_verified" / "validation",
        "prefix": "swe_bench_",
    },
    "mle_bench": {
        "validation_dir": EVALS_ROOT / "mle_bench" / "validation",
        "prefix": "mle_bench_",
    },
    "mlrc_bench": {
        "validation_dir": EVALS_ROOT / "mlrc_bench" / "validation",
        "prefix": "mlrc_bench_",
    },
    "terminal_bench": {
        "validation_dir": EVALS_ROOT / "terminal-bench-2.0" / "validation",
        "prefix": "",
    },
}

## 1. Load and aggregate all validation CSVs

Each CSV has columns `id` and `target` (and optionally `predicate`).
We normalize every file into long-form rows with: `transcript_id`, `benchmark`,
`label_name`, `criteria`, `labeler`, and `target`.

In [10]:
import re

VALID_CRITERIA = {"oh1","oh2","ob3", "t2", "t5"}  # fill in your full set

def parse_label_name(stem: str, prefix: str) -> dict:
    name = stem
    if name.startswith(prefix):
        name = name[len(prefix):]

    # Try to match trailing _INITIALS (2-3 uppercase letters)
    m = re.match(r"^(.+)_([A-Z]{2,3})$", name)
    if m:
        body, labeler = m.group(1), m.group(2)
    else:
        body, labeler = name, "unknown"

    # Match against known criteria anywhere in the body segments
    parts = body.split("_")
    criteria = next((p for p in parts if p in VALID_CRITERIA), parts[-1])

    return {"label_name": name, "criteria": criteria, "labeler": labeler}



rows = []

for bench_name, cfg in VALIDATION_SOURCES.items():
    val_dir = Path(cfg["validation_dir"])
    prefix = cfg.get("prefix", "")
    excludes = cfg.get("exclude_patterns", [])

    for csv_path in sorted(val_dir.rglob("*.csv")):
        if any(pat in csv_path.stem for pat in excludes):
            continue
        df = pd.read_csv(csv_path)
        if "id" not in df.columns or "target" not in df.columns:
            continue

        info = parse_label_name(csv_path.stem, prefix)
        for _, row in df.iterrows():
            rows.append({
                "transcript_id": row["id"],
                "benchmark": bench_name,
                "label_file": csv_path.stem,
                "label_name": info["label_name"],
                "criteria": info["criteria"],
                "labeler": info["labeler"],
                "target": row["target"],
            })

labels_long = pd.DataFrame(rows)

# # Normalize target to numeric (0/1)
# labels_long["target"] = labels_long["target"].map(
#     lambda v: int(v) if str(v) in ("0", "1") else int(bool(v))
# )

print(f"Total label rows: {len(labels_long)}")
print(f"Unique transcripts labeled: {labels_long['transcript_id'].nunique()}")
print(f"Label files loaded: {labels_long['label_file'].nunique()}")
labels_long.head(10)

Total label rows: 1079
Unique transcripts labeled: 343
Label files loaded: 19


,transcript_id,benchmark,label_file,label_name,criteria,labeler,target
0,2HMRtkCxgju7q87nBz5Msm,core_bench,core_bench_easy_oa1_JM,easy_oa1_JM,oa1,JM,0
1,2grUTB3MXmZJW3QcxUKygu,core_bench,core_bench_easy_oa1_JM,easy_oa1_JM,oa1,JM,0
2,4LWaZXamgv9M6GVrA78zgv,core_bench,core_bench_easy_oa1_JM,easy_oa1_JM,oa1,JM,0
3,5QHujhrnsNuSEfThXSukZi,core_bench,core_bench_easy_oa1_JM,easy_oa1_JM,oa1,JM,0
4,86R7tfNjB4UxNEbT2FXva2,core_bench,core_bench_easy_oa1_JM,easy_oa1_JM,oa1,JM,0
5,AGLCdV4EYk534irmHdooSr,core_bench,core_bench_easy_oa1_JM,easy_oa1_JM,oa1,JM,0
6,CqFpbjm5SccfTZrJjHkVvz,core_bench,core_bench_easy_oa1_JM,easy_oa1_JM,oa1,JM,1
7,DUEv7y7yeLCCShLw8YjEhS,core_bench,core_bench_easy_oa1_JM,easy_oa1_JM,oa1,JM,0
8,DYVojGUbAV775LVgkvNPhj,core_bench,core_bench_easy_oa1_JM,easy_oa1_JM,oa1,JM,0
9,EidhWXq6yh3JsQSa2DfTyT,core_bench,core_bench_easy_oa1_JM,easy_oa1_JM,oa1,JM,1


## 2. Summary by benchmark and label file

In [11]:
summary = (
    labels_long.groupby(["benchmark", "label_name", "criteria", "labeler","label_file"])
    .agg(
        n_transcripts=("transcript_id", "nunique"),
        positive_rate=("target", lambda x: (x >= 1).mean()),
        positive_rate_clear=("target", lambda x: (x >= 2).mean()),
    )
    .reset_index()
    .sort_values(["benchmark", "criteria", "labeler", "label_file"])
)

display(summary.style.format({"positive_rate": "{:.0%}","positive_rate_clear": "{:.0%}" }).hide(axis="index"))

benchmark,label_name,criteria,labeler,label_file,n_transcripts,positive_rate,positive_rate_clear
core_bench,easy_oa1_JM,oa1,JM,core_bench_easy_oa1_JM,45,9%,4%
core_bench,easy_ob3_AH,ob3,AH,core_bench_easy_ob3_AH,45,33%,16%
core_bench,easy_ob3_JM,ob3,JM,core_bench_easy_ob3_JM,45,33%,22%
core_bench,easy_oh1_AH,oh1,AH,core_bench_easy_oh1_AH,45,100%,64%
core_bench,easy_oh1_JM,oh1,JM,core_bench_easy_oh1_JM,45,60%,9%
core_bench,easy_oh2_AH,oh2,AH,core_bench_easy_oh2_AH,45,4%,0%
core_bench,easy_oh2_JM,oh2,JM,core_bench_easy_oh2_JM,45,31%,18%
core_bench,medium_t2_JM,t2,JM,core_bench_medium_t2_JM,45,40%,0%
core_bench,medium_t5_JM,t5,JM,core_bench_medium_t5_JM,45,44%,33%
mle_bench,t2_DS,t2,DS,mle_bench_t2_DS,63,11%,0%


## 3. Coverage: transcripts per benchmark and criteria

In [ ]:
import matplotlib.pyplot as plt

coverage = (
    labels_long.groupby(["benchmark", "criteria"])["transcript_id"]
    .nunique()
    .reset_index(name="n_transcripts")
    .sort_values("n_transcripts")
)
coverage["label"] = coverage["benchmark"] + " / " + coverage["criteria"]

fig, ax = plt.subplots(figsize=(8, max(3, 0.45 * len(coverage))))
ax.barh(coverage["label"], coverage["n_transcripts"], color="#4393c3")
ax.set_xlabel("Unique transcripts labeled")
ax.set_title("Human label coverage by benchmark & criteria")
for i, row in coverage.iterrows():
    ax.text(row["n_transcripts"] + 0.5, row["label"], str(row["n_transcripts"]), va="center")
fig.tight_layout()
plt.show()

## 4. Inter-labeler agreement (where multiple labelers exist)

In [12]:
# Find (benchmark, criteria) pairs with multiple labelers
multi_labeler = (
    labels_long.groupby(["benchmark", "criteria"])["labeler"]
    .nunique()
    .reset_index(name="n_labelers")
    .query("n_labelers > 1")
)

if multi_labeler.empty:
    print("No criteria have multiple labelers.")
else:
    for _, ml_row in multi_labeler.iterrows():
        bench, criteria = ml_row["benchmark"], ml_row["criteria"]
        subset = labels_long[
            (labels_long["benchmark"] == bench) & (labels_long["criteria"] == criteria)
        ]
        # Pivot to wide: one column per labeler
        wide = subset.pivot_table(
            index="transcript_id", columns="labeler", values="target", aggfunc="first"
        )
        labelers = wide.columns.tolist()
        agree = (wide[labelers[0]] == wide[labelers[1]]).mean()
        print(f"{bench} / {criteria}: labelers={labelers}, agreement={agree:.0%} (n={len(wide)})")

core_bench / ob3: labelers=['AH', 'JM'], agreement=69% (n=45)
core_bench / oh1: labelers=['AH', 'JM'], agreement=9% (n=45)
core_bench / oh2: labelers=['AH', 'JM'], agreement=69% (n=45)


## 5. Pivot to wide format and save

In [13]:
# Wide format: one row per (transcript, benchmark), one column per label_name
labels_wide = labels_long.pivot_table(
    index=["transcript_id", "benchmark"],
    columns="label_name",
    values="target",
    aggfunc="first",
).reset_index()
labels_wide.columns.name = None

print(f"Wide table: {len(labels_wide)} transcripts x {len(labels_wide.columns)} columns")
display(labels_wide.head(10))

labels_wide.to_csv("preprocessed_human_labels.csv", index=False)
labels_long.to_csv("preprocessed_human_labels_long.csv", index=False)
print("\nSaved preprocessed_human_labels.csv (wide) and preprocessed_human_labels_long.csv (long)")

Wide table: 343 transcripts x 19 columns


,transcript_id,benchmark,easy_oa1_JM,easy_ob3_AH,easy_ob3_JM,easy_oh1_AH,easy_oh1_JM,easy_oh2_AH,easy_oh2_JM,medium_t2_JM,medium_t5_JM,oh1_JM,oh1_terminal_bench_2,oh2_terminal_bench_2,t2_DS,t2_terminal_bench_2,t5_DS,t5_JM,t5_terminal_bench_2
0,2HMRtkCxgju7q87nBz5Msm,core_bench,0,3,2,2,1,0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2HaDwauAP9WqyjDc4so5vs,swe_bench,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,0,NaN
2,2REPPigj8zSdTH5E8CLj96,terminal_bench,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1,NaN,0,NaN,NaN,0
3,2V6KZjYvE8gY4iTLNsA4Rh,swe_bench,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,0,NaN
4,2VKcpLq4tJULmQTnqkQgah,core_bench,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2VdTQfuNtUjzFXqJTJrSRi,core_bench,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2XjrxR6hSqHrHntybyxwcd,terminal_bench,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,0,NaN,NaN,0
7,2euKmsNJffEFhfybrb9VbY,mle_bench,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,0,NaN,NaN
8,2grUTB3MXmZJW3QcxUKygu,core_bench,0,2,1,2,0,0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2osMwK82oHDoYxwpg6j4K4,mle_bench,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,0,NaN,NaN



Saved preprocessed_human_labels.csv (wide) and preprocessed_human_labels_long.csv (long)
